# Phase 2.1 -- Content Based Filtering Model
Each user has games that they have previously liked, denoted by `voted_up` = True. This model recommends games to each user based on similarities to the games that the user has already liked, using game metadata and game descriptions to determine similarities.

---
Inputs from:
*   Phase 0: Game static features, BERT description embeddings
*   Phase 1: NCF model output (hits and misses)



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/Phase 2/Phase 2 data/data'

In [3]:
import pandas as pd
import numpy as np
import os, random
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity
import pickle

## 1. Load data

In [5]:
game_features = pd.read_parquet('/content/drive/MyDrive/feature_engineered_data/game_features_static.parquet')
game_embeddings = pd.read_parquet('/content/drive/MyDrive/feature_engineered_data/game_description_embeddings.parquet')
train = pd.read_parquet('/content/drive/MyDrive/feature_engineered_data/train_with_features.parquet')
test = pd.read_parquet('/content/drive/MyDrive/feature_engineered_data/test_with_features.parquet')

## 2. Construct feature vector

Merge the static game features with the corresponding game description embeddings.

In [6]:
game_vector = pd.merge(game_features, game_embeddings, on='appid')
game_vector = game_vector.reset_index()
print(game_vector.shape)

(32883, 223)


In [7]:
print(game_vector.columns.tolist())

['appid', 'is_free', 'required_age', 'metacritic_score', 'mat_final_price_log', 'genre_Accounting', 'genre_Action', 'genre_Adventure', 'genre_Animation & Modeling', 'genre_Audio Production', 'genre_Casual', 'genre_Design & Illustration', 'genre_Early Access', 'genre_Education', 'genre_Free To Play', 'genre_Game Development', 'genre_Gore', 'genre_Indie', 'genre_Massively Multiplayer', 'genre_Movie', 'genre_Nudity', 'genre_Photo Editing', 'genre_RPG', 'genre_Racing', 'genre_Sexual Content', 'genre_Simulation', 'genre_Software Training', 'genre_Sports', 'genre_Strategy', 'genre_Utilities', 'genre_Video Production', 'genre_Violent', 'genre_Web Publishing', 'cat_Adjustable Difficulty', 'cat_Adjustable Text Size', 'cat_Camera Comfort', 'cat_Captions available', 'cat_Chat Speech-to-text', 'cat_Chat Text-to-speech', 'cat_Co-op', 'cat_Color Alternatives', 'cat_Commentary available', 'cat_Cross-Platform Multiplayer', 'cat_Custom Volume Controls', 'cat_Family Sharing', 'cat_Full controller suppor

## 3. Feature matrix construction
Builds **two** feature matrixes: `static_matrix`, representing each game based on game metadata, and `emb_matrix`, representing each game based on game description embeddings.

For each game in `game_vector`:
1. The genre & category columns are standardized separately, then combined and normalized to construct `static_matrix`, a feature matrix **with only game metadata.**
2. The description embedding columns are normalized to construct `emb_matrix`, a feature matrix **with only game description embeddings.**

In [8]:
genre_cols = [c for c in game_vector.columns if c.startswith('genre_')]
cat_cols = [c for c in game_vector.columns if c.startswith('cat_')]
emb_cols = [c for c in game_vector.columns if c.startswith('desc_emb_')]

scaler = StandardScaler()
genre_values = scaler.fit_transform(game_vector[genre_cols].values)
cat_values = scaler.fit_transform(game_vector[cat_cols].values)
emb_values = scaler.fit_transform(game_vector[emb_cols].values)

## Static features matrix
static_matrix = normalize(np.hstack([
    genre_values,
    cat_values
]))
print(f"Static matrix shape: {static_matrix.shape}")
## Description embedding matrix
emb_matrix = normalize(emb_values)
print(f"Embedding matrix shape: {emb_matrix.shape}")


Static matrix shape: (32883, 86)
Embedding matrix shape: (32883, 128)


## 4. Defining evaluation metrics

`hr_at_k`: Checks whether at least one of the actual liked games appears in the top K recommended games.

`ndcg_at_k`: Evaluates how accurately recommendations are ranked.


---


Overall evaluation metrics used:
- **HR@10** — did any recommended game appear in the user's other liked games?
- **NDCG@10** — how highly ranked were the relevant games?
- **Coverage** — what fraction of the catalogue was recommended across all users?

In [9]:
def hr_at_k(recs, actual_ids, k=10):
  return int(any(i in recs[:k] for i in actual_ids))

def ndcg_at_k(recs, actual_ids, k=10):
  for rank, rec in enumerate(recs[:k]):
    if rec in actual_ids:
      return 1 / np.log2(rank + 2)
  return 0.0

## 5. Running the model

### Final model

Each user's most recently liked game was held out as the test item, with their remaining liked games used to build a weighted user profile.
`static_matrix` and `emb_matrix` were used to compute final similarity scores by blending static and embedding similarity scores with `beta=0.4`.

For each user in `train_loo`:
1. Iterates over their liked games, computing a **recency-weighted** cosine similarity score against all games in both `static_matrix` and `emb_matrix`. More recently liked games contribute more to the user's profile.
2. Zeroes out already-liked games to avoid recommending them.
3. Blends the two scores using `beta` to produce `final_scores`.
4. Stores the **top 500 candidates** per user (by score) in `all_recs_scored` for use in downstream ranking stages.
5. Evaluates the top 10 recommendations against the held-out game using **HR@10** and **NDCG@10**.

**Results:** HR@10: 0.0563, NDCG@10: 0.0373, Coverage: 95.59%

**This was the best result out of all tests.

In [ ]:
beta = 0.4
results = []
all_recs_scored = {}
train_loo = {}
test_loo = {}
hr_scores = []
ndcg_scores = []
all_recs = {}

train_sorted = train[train['voted_up'] == True].sort_values(by='created_at')
train_liked = train_sorted.groupby('author_steamid')['appid'].apply(list)
test_liked = test[test['voted_up'] == True].groupby('author_steamid')['appid'].apply(list)

appids = game_vector['appid'].values
appid_idx = {appid: i for i, appid in enumerate(appids)}

for user_id, games in train_liked.items():
  if len(games) < 2:
    continue
  train_loo[user_id] = games[:-1]
  test_loo[user_id] = [games[-1]]

for user_id, liked_games in tqdm(train_loo.items(), total=len(train_loo)):
  static_score = np.zeros(len(appids))
  emb_score = np.zeros(len(appids))

  for rank, game in enumerate(liked_games):
    if game not in appid_idx:
      continue
    weight = (rank + 1) / len(liked_games)
    idx = appid_idx[game]
    ## Static game features
    liked_static_game = static_matrix[idx]
    static_score += weight * (static_matrix @ liked_static_game)

    ## Description embedding
    liked_emb_game = emb_matrix[idx]
    emb_score += weight * (emb_matrix @ liked_emb_game)

  for game in liked_games:
    if game in appid_idx:
      game_idx = appid_idx[game]
      static_score[game_idx] = -1
      emb_score[game_idx] = -1

  final_scores = beta * static_score + (1 - beta) * emb_score

  liked_set = set(liked_games)
  top_indices = np.argsort(final_scores)[::-1]
  top_scored = {}
  count = 0
  for i in top_indices:
    if appids[i] not in liked_set:
      top_scored[appids[i]] = final_scores[i]
      count += 1
      if count >= 500:
          break
  all_recs_scored[user_id] = top_scored

  ## Rank the games by similarity scores
  top_scores = np.argsort(final_scores)[::-1][:10]
  recs = [appids[i] for i in top_scores]
  all_recs[user_id] = recs

  target = [test_loo[user_id][0]]
  hr_scores.append(hr_at_k(recs, target))
  ndcg_scores.append(ndcg_at_k(recs, target))

all_recommended = set(appid for recs in all_recs.values() for appid in recs)
coverage = len(all_recommended) / len(appids) * 100
print(f"beta={beta:.1f} (struct={beta:.0%}, emb={1-beta:.0%}) --> HR@10: {np.mean(hr_scores):.4f}, NDCG@10: {np.mean(ndcg_scores):.4f},  coverage: {coverage}")

results.append({
  'beta': beta,
  'HR@10': np.mean(hr_scores),
  'NDCG@10': np.mean(ndcg_scores),
  'coverage': coverage
})

best_result = max(results, key=lambda x: x['HR@10'])
print(f"\nBest: {best_result}")

print(f'\nHR@10: {best_result['HR@10']:.4f}')
print(f'NDCG@10: {best_result['NDCG@10']:.4f}')
print(f'Coverage: {best_result['coverage']:.4f}')


In [ ]:
## Save all_recs_scored (model with beta=0.4)
with open(os.path.join(OUTPUT_DIR, 'final_all_recs_scored.pkl'), 'wb') as f:
    pickle.dump(all_recs_scored, f)

## Save model
cbf_model = {
  'static_matrix': static_matrix,
  'emb_matrix': emb_matrix,
  'appids': appids,
  'appid_idx': appid_idx,
  'beta': 0.4
}
with open(os.path.join(OUTPUT_DIR, 'final_cbf_model.pkl'), 'wb') as f:
    pickle.dump(cbf_model, f)

In [ ]:
## Load final_all_recs_scored
with open(os.path.join(OUTPUT_DIR, 'final_all_recs_scored.pkl'), 'rb') as f:
  final_all_recs_scored = pickle.load(f)

## Load final_cbf_model
with open(os.path.join(OUTPUT_DIR, 'final_cbf_model.pkl'), 'rb') as f:
  final_cbf_model = pickle.load(f)

## 6. Evaluating CBF model's value-add to Phase 1's NCF model

In [ ]:
rows = []
for user_id, recs in tqdm(final_all_recs_scored.items()):
  ranked = sorted(recs.items(), key=lambda x: x[1], reverse=True)
  for rank, (appid, score) in enumerate(ranked, 1):
    rows.append({
        'author_steamid': user_id,
        'appid': appid,
        'content_sim_score': score,
        'rank': rank
    })

content_similarity_scores = pd.DataFrame(rows)
content_similarity_scores.to_parquet(os.path.join(OUTPUT_DIR, 'phase2_content_similarity_scores.parquet'))

100%|██████████| 25305/25305 [00:10<00:00, 2425.98it/s]


      author_steamid    appid  content_sim_score  rank
0  76561197960272177  1637520           0.469213     1
1  76561197960272177  2915020           0.446533     2
2  76561197960272177  2410420           0.441852     3
3  76561197960272177   606880           0.437013     4
4  76561197960272177  1882280           0.435305     5


In [ ]:
ncf_hits = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Phase 2/ncf parquets/ncf_test_hits.parquet')
ncf_misses = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Phase 2/ncf parquets/ncf_test_misses.parquet')

## Given a user & a positive item (that NCF failed on), check its corresponding rank in the CBF recommendations
def get_cbf_rank(user_id, pos_appid, cbf_recs):
  user_recs = cbf_recs.get(user_id) or cbf_recs.get(int(user_id)) or cbf_recs.get(str(user_id))
  if user_recs is None or pos_appid not in user_recs:
    return None
  ranked = sorted(user_recs.items(), key=lambda x: x[1], reverse=True)
  for rank, (appid, _) in enumerate(ranked, 1):
      if appid == pos_appid:
          return rank
  return None

for df, label in [(ncf_misses, "miss"), (ncf_hits, "hit")]:
  df["cbf_rank"] = df.apply(
    lambda r: get_cbf_rank(r["author_steamid"], r["appid"], final_all_recs_scored),
    axis=1
  )


In [ ]:
total_miss = len(ncf_misses)
total_hit = len(ncf_hits)
total_test = total_miss + total_hit

surfaced_misses = ncf_misses["cbf_rank"].notna()
print(f"Total NCF misses: {total_miss:,}")
print(f"CBF surfaced positive at all: {surfaced_misses.sum():,} ({surfaced_misses.mean():.1%})")
print(f"CBF HR@10 on NCF-misses: {(ncf_misses['cbf_rank'] <= 10).mean():.3f}")
print(f"Median CBF rank (when surfaced): {ncf_misses['cbf_rank'].median():.0f}")

rescued = (ncf_misses["cbf_rank"] <= 10).sum()
print(f"\nOf all NCF misses, CBF found: {rescued:,} / {total_miss:,} ({rescued/total_miss:.1%} of misses by NCF)")

Total NCF misses: 2,545
CBF surfaced positive at all: 270 (10.6%)
CBF HR@10 on NCF-misses: 0.009
Median CBF rank (when surfaced): 196

Of all NCF misses, CBF found: 23 / 2,545 (0.9% of misses by NCF)


**Results:**
- Game metadata alone is insufficient for building a robust content-based recommender. Hit rate and NDCG are low (< 0.1)
- The best CBF model (beta=0.4) does not improve on the NCF model's results. Out of all the NCF model's misses, CBF only surfaced 0.9%.



## 7. Additional Tests (Improving CBF)

### 7.1. Test 1: LOO with weights (Baseline)

Each user's most recently liked game is held out as the test item, while their remaining liked games are used to build a weighted user profile. Different weighting methods were tried.

Recommendations are generated by computing cosine similarity between the user profile and all games, excluding already liked items. The recommendations are then evaluated using HR@10 and NDCG@10.

#### 7.1.1. Feature matrix construction (Alternative Step 3)
Builds a normalized feature matrix representing each game based on combined metadata and description embeddings.

For each game in `game_vector`, the genre, category and description embedding columns are standardized, then combined to construct a single array. This creates `feature_vector`, a single feature matrix.

In [ ]:
genre_cols = [c for c in game_vector.columns if c.startswith('genre_')]
cat_cols = [c for c in game_vector.columns if c.startswith('cat_')]
emb_cols = [c for c in game_vector.columns if c.startswith('desc_emb_')]

scaler = StandardScaler()
genre_values = scaler.fit_transform(game_vector[genre_cols].values)
cat_values = scaler.fit_transform(game_vector[cat_cols].values)
emb_values = scaler.fit_transform(game_vector[emb_cols].values)

feature_matrix = np.hstack([
    genre_values,
    cat_values,
    emb_values
])
print(f'Feature matrix shape: {feature_matrix.shape}')
feature_matrix = normalize(feature_matrix)
feature_matrix = feature_matrix.astype(np.float32)

appids = game_vector['appid'].values
appid_idx = {appid: i for i, appid in enumerate(appids)}


Feature matrix shape: (32883, 214)


In [ ]:
train_loo = {}
test_loo = {}
hr_scores = []
ndcg_scores = []
all_recs = {}
all_recs_scored = {}

train_sorted = train[train['voted_up'] == True].sort_values(by='created_at')
train_liked = train_sorted.groupby('author_steamid')['appid'].apply(list)
test_liked = test[test['voted_up'] == True].groupby('author_steamid')['appid'].apply(list)

for user_id, games in train_liked.items():
  if len(games) < 2:
    continue
  train_loo[user_id] = games[:-1]
  test_loo[user_id] = [games[-1]]

for user_id, liked_games in tqdm(train_loo.items(), total=len(train_loo)):
  scores = np.zeros(len(appids))
  ## For each user's liked games:
  ## - Get the weight of each game
  ## - Multiply the weight to the similarity score, such that later games have higher weight
  for rank, game in enumerate(liked_games):
    if game not in appid_idx:
      continue
    weight = (rank + 1) / len(liked_games) ## give earlier liked games lower weight
    idx = appid_idx[game]
    liked_game = feature_matrix[idx]
    scores += weight * (feature_matrix @ liked_game) ## feature_matrix * liked_game row == cosine similarity

  ## Zero the games that were already liked by the user
  for game in liked_games:
    if game in appid_idx:
      game_idx = appid_idx[game]
      scores[game_idx] = -1

  ## Rank the games by similarity scores
  top_scores = np.argsort(scores)[::-1][:10]
  recs = [appids[i] for i in top_scores]

  liked_set = set(liked_games)
  top_indices = np.argsort(scores)[::-1]
  top_scored = {}
  count = 0
  for i in top_indices:
      if appids[i] not in liked_set:
          top_scored[appids[i]] = scores[i]
          count += 1
          if count >= 500:
              break
  all_recs_scored[user_id] = top_scored
  all_recs[user_id] = recs

  target = [test_loo[user_id][0]]
  hr_scores.append(hr_at_k(recs, target))
  ndcg_scores.append(ndcg_at_k(recs, target))

all_recommended = set(appid for recs in all_recs.values() for appid in recs)
coverage = len(all_recommended) / len(appids) * 100

print(f'\nHR@10: {np.mean(hr_scores):.4f}')
print(f'NDCG@10: {np.mean(ndcg_scores):.4f}')
print(f'Coverage: {coverage:.4f}')


  0%|          | 14/25305 [00:00<07:33, 55.82it/s]


KeyboardInterrupt: 

#### Result of testing weighting: How strongly recency is emphasized has very little influence on the model

`weight = np.exp((rank + 1) / len(liked_games))`
- Most recent items get significantly higher weight
- Results:
  - HR@10: 0.0475
  - NDCG@10: 0.0329
  - Coverage: 95.0217


`weight = np.exp(-0.1 * (len(liked_games) - 1 - rank))`
- Exponential decay -- recent items get slightly higher weight
- Results:
  - HR@10: 0.0472
  - NDCG@10: 0.0325
  - Coverage: 95.1951

`weight = (rank + 1) / len(liked_games)`
- Increasing weight linearly over time -- mildest recency bias
- Results:
  - HR@10: 0.0475
  - NDCG@10: 0.0328
  - Coverage: 95.0339

In [ ]:
## Save all_recs && all_recs_scored
with open(os.path.join(OUTPUT_DIR, 'all_recs.pkl'), 'wb') as f:
    pickle.dump(all_recs, f)

with open(os.path.join(OUTPUT_DIR, 'all_recs_scored.pkl'), 'wb') as f:
    pickle.dump(all_recs_scored, f)

In [ ]:
# Load all_recs && all_recs_scored
with open(os.path.join(OUTPUT_DIR, 'all_recs.pkl'), 'rb') as f:
    all_recs = pickle.load(f)

with open(os.path.join(OUTPUT_DIR, 'all_recs_scored.pkl'), 'rb') as f:
    all_recs_scored = pickle.load(f)

### 7.2. Test 2: Changing weights for static game features/description embeddings

From Step 3, the static game features are outnumbered by description embeddings. As such, the embeddings may dominate the cosine similarity, causing it to become unbalanced. This test will check if upweighting the static game features in response can improve HR@10 & NDCG@10.

Results will be stored in `alternate_weights_results.parquet`.

In [ ]:
results_sweep = []

train_loo = {}
test_loo = {}

train_sorted = train[train['voted_up'] == True].sort_values(by='created_at')
train_liked = train_sorted.groupby('author_steamid')['appid'].apply(list)
test_liked = test[test['voted_up'] == True].groupby('author_steamid')['appid'].apply(list)

appids = game_vector['appid'].values
appid_idx = {appid: i for i, appid in enumerate(appids)}


## Replaces Step 2: Feature matrix construction
genre_values = scaler.fit_transform(game_vector[genre_cols].values)
cat_values = scaler.fit_transform(game_vector[cat_cols].values)
emb_values = scaler.fit_transform(game_vector[emb_cols].values)

for user_id, games in train_liked.items():
  if len(games) < 2:
    continue
  train_loo[user_id] = games[:-1]
  test_loo[user_id] = [games[-1]]

for static_weight in [0.5, 1.0, 2.0, 3.0, 5.0]:
  weighted_matrix = np.hstack([
      genre_values * static_weight,
      cat_values * static_weight,
      emb_values * 1.0
  ])
  print(f'Feature matrix shape with the weight {static_weight} applied: {weighted_matrix.shape}')
  weighted_matrix = normalize(weighted_matrix)

  hr_scores = []
  ndcg_scores = []
  all_recs = {}

  for user_id, liked_games in tqdm(train_loo.items(), total=len(train_loo)):
    scores = np.zeros(len(appids))
    for rank, game in enumerate(liked_games):
      if game not in appid_idx:
        continue
      weight = (rank + 1) / len(liked_games)
      idx = appid_idx[game]
      liked_game = weighted_matrix[idx]
      scores += weight * (weighted_matrix @ liked_game)

    ## Zero the games that were already liked by the user
    for game in liked_games:
      if game in appid_idx:
        game_idx = appid_idx[game]
        scores[game_idx] = -1

    ## Rank the games by similarity scores
    top_scores = np.argsort(scores)[::-1][:10]
    recs = [appids[i] for i in top_scores]

    target = [test_loo[user_id][0]]
    hr_scores.append(hr_at_k(recs, target))
    ndcg_scores.append(ndcg_at_k(recs, target))
    all_recs[user_id] = recs

  all_recommended = set(appid for recs in all_recs.values() for appid in recs)
  coverage = len(all_recommended) / len(appids) * 100

  results_sweep.append({
      'static_weight': static_weight,
      'HR@10': np.mean(hr_scores),
      'NDCG@10': np.mean(ndcg_scores),
      'coverage': coverage
  })

best_result = max(results_sweep, key=lambda x: x['HR@10'])
print(f"\nBest: {best_result}")

print(f'\nHR@10: {np.mean(best_result['HR@10']):.4f}')
print(f'NDCG@10: {np.mean(best_result['NDCG@10']):.4f}')
print(f'Coverage: {best_result['coverage']:.4f}')


Feature matrix shape with the weight 0.5 applied: (32883, 214)


100%|██████████| 25305/25305 [12:07<00:00, 34.77it/s]


Feature matrix shape with the weight 1.0 applied: (32883, 214)


100%|██████████| 25305/25305 [11:46<00:00, 35.84it/s]


Feature matrix shape with the weight 2.0 applied: (32883, 214)


...

Feature matrix shape with the weight 3.0 applied: (32883, 214)


100%|██████████| 25305/25305 [11:59<00:00, 35.17it/s]


Feature matrix shape with the weight 5.0 applied: (32883, 214)


100%|██████████| 25305/25305 [12:12<00:00, 34.54it/s]



Best: {'static_weight': 2.0, 'HR@10': np.float64(0.05303299743133768), 'NDCG@10': np.float64(0.035720427945977094), 'coverage': 93.91478879664264}

HR@10: 0.0530
NDCG@10: 0.0357
Coverage: 93.9148


In [ ]:
results_sweep_df = pd.DataFrame(results_sweep)
results_sweep_df.columns = ['weight', 'HR@10', 'NDCG@10', 'coverage']
results_sweep_df.to_parquet(os.path.join(OUTPUT_DIR, 'alternate_weights_results.parquet'))

print(results_sweep_df)

   weight     HR@10   NDCG@10   coverage
0     0.5  0.042087  0.028905  95.946234
1     1.0  0.047461  0.032807  95.036949
2     2.0  0.053033  0.035720  93.914789
3     3.0  0.051650  0.034539  93.181887
4     5.0  0.046315  0.030526  92.147918


### 7.3. Test 3: Ranking 1 positive against 99 random negatives

Due to the sparsity of most users' liked games, it is hard to accurately predict and rank recommendations against all other games in the dataset. As such, this test will measure if a user's true item ranks in the top 10 when competing against 99 randomly sampled negative items (hit rate), instead of all games possible.

In [ ]:
all_game_ids = set(appids)
sampled_hits = []
train_loo = {}
test_loo = {}

train_sorted = train[train['voted_up'] == True].sort_values(by='created_at')
train_liked = train_sorted.groupby('author_steamid')['appid'].apply(list)
test_liked = test[test['voted_up'] == True].groupby('author_steamid')['appid'].apply(list)

for user_id, games in train_liked.items():
  if len(games) < 2:
    continue
  train_loo[user_id] = games[:-1]
  test_loo[user_id] = [games[-1]]

for user_id, liked_games in tqdm(train_loo.items(), total=len(train_loo)):
  scores = np.zeros(len(appids))
  for rank, game in enumerate(liked_games):
      if game not in appid_idx:
          continue
      weight = (rank + 1) / len(liked_games)
      idx = appid_idx[game]
      liked_game = feature_matrix[idx]
      scores += weight * (feature_matrix @ liked_game)

  for game in liked_games:
      if game in appid_idx:
          scores[appid_idx[game]] = -1

  ## Create the pool of potential games
  true_id = test_loo[user_id][0]
  liked_set = set(liked_games)
  game_pool = list(all_game_ids - liked_set - {true_id})
  negatives = random.sample(game_pool, min(99, len(game_pool)))
  candidates = set([true_id] + negatives)

  candidate_idx = np.array([appid_idx[g] for g in candidates if g in appid_idx])
  candidate_scores = scores[candidate_idx]
  ranked = candidate_idx[np.argsort(candidate_scores)[::-1]]
  recs_filtered = [appids[i] for i in ranked]
  hit = int(true_id in recs_filtered[:10])

  sampled_hits.append(hit)

print(f"\nSampled HR@10 (1 pos + 99 neg): {np.mean(sampled_hits):.4f}")
print(f"Total users evaluated: {len(sampled_hits)}")


100%|██████████| 25305/25305 [11:19<00:00, 37.22it/s]

Sampled HR@10 (1 pos + 99 neg): 0.3930
Total users evaluated: 25305


### 7.4. Test 4: Calculate cosine similarity separately

This test will calculate cosine similarity **separately** for game static features and for game description embeddings. Different betas are tested, which control the extent to which static game features & game description embeddings affect the total similarity score

(higher beta == static game features have greater influence).  

Results will be stored in `separate_static_emb_results.parquet`.

In [ ]:
## Replaces Step 2: Feature matrix construction
results_sweep = []

genre_values = scaler.fit_transform(game_vector[genre_cols].values)
cat_values = scaler.fit_transform(game_vector[cat_cols].values)
emb_values = scaler.fit_transform(game_vector[emb_cols].values)

## Static features matrix
static_matrix = normalize(np.hstack([
    genre_values,
    cat_values
]))
print(f"Static matrix shape: {static_matrix.shape}")
## Description embedding matrix
emb_matrix = normalize(emb_values)
print(f"Embedding matrix shape: {emb_matrix.shape}")


train_loo = {}
test_loo = {}

train_sorted = train[train['voted_up'] == True].sort_values(by='created_at')
train_liked = train_sorted.groupby('author_steamid')['appid'].apply(list)
test_liked = test[test['voted_up'] == True].groupby('author_steamid')['appid'].apply(list)

appids = game_vector['appid'].values
appid_idx = {appid: i for i, appid in enumerate(appids)}

for user_id, games in train_liked.items():
  if len(games) < 2:
    continue
  train_loo[user_id] = games[:-1]
  test_loo[user_id] = [games[-1]]


## Compute user profiles (static game features & description embedding scores, all liked_games)
user_profiles = {}

for beta in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    print(f"Current beta: {beta}")
    hr_scores = []
    ndcg_scores = []
    all_recs = {}

    for user_id, liked_games in tqdm(train_loo.items(), total=len(train_loo)):
      static_score = np.zeros(len(appids))
      emb_score = np.zeros(len(appids))

      for rank, game in enumerate(liked_games):
        if game not in appid_idx:
          continue
        weight = (rank + 1) / len(liked_games)
        idx = appid_idx[game]
        ## Static game features
        liked_static_game = static_matrix[idx]
        static_score += weight * (static_matrix @ liked_static_game)

        ## Description embedding
        liked_emb_game = emb_matrix[idx]
        emb_score += weight * (emb_matrix @ liked_emb_game)

      for game in liked_games:
        if game in appid_idx:
          game_idx = appid_idx[game]
          static_score[game_idx] = -1
          emb_score[game_idx] = -1

      final_scores = beta * static_score + (1 - beta) * emb_score
      ## Rank the games by similarity scores
      top_scores = np.argsort(final_scores)[::-1][:10]
      recs = [appids[i] for i in top_scores]
      all_recs[user_id] = recs

      target = [test_loo[user_id][0]]
      hr_scores.append(hr_at_k(recs, target))
      ndcg_scores.append(ndcg_at_k(recs, target))

    all_recommended = set(appid for recs in all_recs.values() for appid in recs)
    coverage = len(all_recommended) / len(appids) * 100
    print(f"beta={beta:.1f} (struct={beta:.0%}, emb={1-beta:.0%}) --> HR@10: {np.mean(hr_scores):.4f}, NDCG@10: {np.mean(ndcg_scores):.4f},  coverage: {coverage}")

    results_sweep.append({
      'beta': beta,
      'HR@10': np.mean(hr_scores),
      'NDCG@10': np.mean(ndcg_scores),
      'coverage': coverage
    })

best_result = max(results_sweep, key=lambda x: x['HR@10'])
print(f"\nBest: {best_result}")

print(f'\nHR@10: {np.mean(best_result['HR@10']):.4f}')
print(f'NDCG@10: {np.mean(best_result['NDCG@10']):.4f}')
print(f'Coverage: {best_result['coverage']:.4f}')



Static matrix shape: (32883, 86)
Embedding matrix shape: (32883, 128)
Current beta: 0.0


100%|██████████| 25305/25305 [12:02<00:00, 35.03it/s]


beta=0.0 (struct=0%, emb=100%) --> HR@10: 0.0369, NDCG@10: 0.0247,  coverage: 96.65176534987684
Current beta: 0.2


100%|██████████| 25305/25305 [11:10<00:00, 37.72it/s]


beta=0.2 (struct=20%, emb=80%) --> HR@10: 0.0528, NDCG@10: 0.0354,  coverage: 95.3927561353891
Current beta: 0.4


...

beta=0.6 (struct=60%, emb=40%) --> HR@10: 0.0520, NDCG@10: 0.0346,  coverage: 96.32028707842957
Current beta: 0.8


100%|██████████| 25305/25305 [10:23<00:00, 40.58it/s]


beta=0.8 (struct=80%, emb=20%) --> HR@10: 0.0466, NDCG@10: 0.0314,  coverage: 96.70954596600066
Current beta: 1.0


100%|██████████| 25305/25305 [11:03<00:00, 38.17it/s]


beta=1.0 (struct=100%, emb=0%) --> HR@10: 0.0259, NDCG@10: 0.0142,  coverage: 96.73387464647386

Best: {'beta': 0.4, 'HR@10': np.float64(0.05623394586050188), 'NDCG@10': np.float64(0.03732920797024056), 'coverage': 95.58738557917465}

HR@10: 0.0562
NDCG@10: 0.0373
Coverage: 95.5874


In [ ]:
results_static_emb_df = pd.DataFrame(results_sweep)
results_static_emb_df.columns = ['beta', 'HR@10', 'NDCG@10', 'coverage']
results_static_emb_df.to_parquet(os.path.join(OUTPUT_DIR, 'separate_static_emb_results.parquet'))

print(results_static_emb_df)

   beta     HR@10   NDCG@10   coverage
0   0.0  0.036910  0.024720  96.651765
1   0.2  0.052756  0.035399  95.392756
2   0.4  0.056234  0.037329  95.587386
3   0.6  0.052045  0.034560  96.320287
4   0.8  0.046631  0.031387  96.709546
5   1.0  0.025884  0.014240  96.733875


### 7.5. Test 7: Combining optimal weight & beta

This test will combine `static_weight = 2` (from Test 2) and `beta = 0.4` (from Test 4).

Results will be stored in `combined_results.parquet`.

In [ ]:
## Replaces Step 2: Feature matrix construction
optimized_results = []

genre_values = scaler.fit_transform(game_vector[genre_cols].values)
cat_values = scaler.fit_transform(game_vector[cat_cols].values)
emb_values = scaler.fit_transform(game_vector[emb_cols].values)

static_weight = 2.0
beta = 0.4
print(f"Weights (static features): {static_weight}")
print(f"Beta: {beta}")

## Static features matrix
static_matrix = normalize(np.hstack([
    genre_values * static_weight,
    cat_values * static_weight
]))
print(f"Static matrix shape: {static_matrix.shape}")
## Description embedding matrix
emb_matrix = normalize(emb_values)
print(f"Embedding matrix shape: {emb_matrix.shape}")


train_loo = {}
test_loo = {}
hr_scores = []
ndcg_scores = []
all_recs = {}

train_sorted = train[train['voted_up'] == True].sort_values(by='created_at')
train_liked = train_sorted.groupby('author_steamid')['appid'].apply(list)
test_liked = test[test['voted_up'] == True].groupby('author_steamid')['appid'].apply(list)

appids = game_vector['appid'].values
appid_idx = {appid: i for i, appid in enumerate(appids)}

for user_id, games in train_liked.items():
  if len(games) < 2:
    continue
  train_loo[user_id] = games[:-1]
  test_loo[user_id] = [games[-1]]

for user_id, liked_games in tqdm(train_loo.items(), total=len(train_loo)):
  static_score = np.zeros(len(appids))
  emb_score = np.zeros(len(appids))

  for rank, game in enumerate(liked_games):
    if game not in appid_idx:
      continue
    weight = (rank + 1) / len(liked_games)
    idx = appid_idx[game]
    ## Static game features
    liked_static_game = static_matrix[idx]
    static_score += weight * (static_matrix @ liked_static_game)

    ## Description embedding
    liked_emb_game = emb_matrix[idx]
    emb_score += weight * (emb_matrix @ liked_emb_game)

  for game in liked_games:
    if game in appid_idx:
      game_idx = appid_idx[game]
      static_score[game_idx] = -1
      emb_score[game_idx] = -1

  final_scores = beta * static_score + (1 - beta) * emb_score
  ## Rank the games by similarity scores
  top_scores = np.argsort(final_scores)[::-1][:10]
  recs = [appids[i] for i in top_scores]
  all_recs[user_id] = recs

  target = [test_loo[user_id][0]]
  hr_scores.append(hr_at_k(recs, target))
  ndcg_scores.append(ndcg_at_k(recs, target))

all_recommended = set(appid for recs in all_recs.values() for appid in recs)
coverage = len(all_recommended) / len(appids) * 100
print(f"beta={beta:.1f} (struct={beta:.0%}, emb={1-beta:.0%}) --> HR@10: {np.mean(hr_scores):.4f}, NDCG@10: {np.mean(ndcg_scores):.4f},  coverage: {coverage}")

optimized_results.append({
  'HR@10': np.mean(hr_scores),
  'NDCG@10': np.mean(ndcg_scores),
  'coverage': coverage
})

print(f"\nResults (weight = {static_weight} & beta = {beta})")
print(f'\nHR@10: {optimized_results[0]['HR@10']:.4f}')
print(f'NDCG@10: {optimized_results[0]['NDCG@10']:.4f}')
print(f'Coverage: {optimized_results[0]['coverage']:.4f}')



Weights (static features): 2.0
Beta: 0.4
Static matrix shape: (32883, 86)
Embedding matrix shape: (32883, 128)


100%|██████████| 25305/25305 [10:53<00:00, 38.70it/s]


beta=0.4 (struct=40%, emb=60%) --> HR@10: 0.0562, NDCG@10: 0.0373,  coverage: 95.58738557917465

Results (weight = 2.0 & beta = 0.4)

HR@10: 0.0562
NDCG@10: 0.0373
Coverage: 95.5874


In [ ]:
results_combined_df = pd.DataFrame(optimized_results)
results_combined_df.columns = ['HR@10', 'NDCG@10', 'coverage']
results_combined_df.to_parquet(os.path.join(OUTPUT_DIR, 'combined_results.parquet'))

print(results_combined_df)

      HR@10   NDCG@10   coverage
0  0.056234  0.037329  95.587386
